<a href="https://colab.research.google.com/github/kasturikirankumar1101-lab/AI_TOOLS/blob/main/AnyLangCodeConverter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Code Converter — Production-Grade Gradio App
Converts source code between languages using OpenAI GPT API.
Runs in Google Colab (reads key from Colab Secrets) or locally (reads from env var).
"""
#--------------------------------------------------------------------
# This program uses the GPT closed source model by caling API
# ___________________________________________________________________


import os
import re
import logging
import gradio as gr
from openai import OpenAI, APIError, APIConnectionError, RateLimitError, AuthenticationError

# ── Colab Secrets (preferred) with os.environ fallback ────────────────────────
try:
    from google.colab import userdata as colab_userdata
    _COLAB = True
except ImportError:
    _COLAB = False

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

if _COLAB:
    logger.info("Running inside Google Colab — API key will be read from Colab Secrets.")
else:
    logger.info("Running outside Google Colab — API key will be read from environment variable.")

# ── Constants ──────────────────────────────────────────────────────────────────
LANGUAGES      = ["Java", "Python", "Javascript", "C++", "COBOL", "VC++", ".NET"]
MODEL          = "gpt-4o"    # change to "gpt-4-turbo" or "gpt-3.5-turbo" if needed
MAX_TOKENS     = 8192        # enough for large file conversions
MAX_INPUT_CHARS = 100_000    # ~100 KB safety cap on input


# ── API Client ─────────────────────────────────────────────────────────────────
def get_client() -> OpenAI:
    """
    Resolve the OpenAI API key and return an OpenAI client.

    Resolution order:
      1. Google Colab Secrets  (key name: OPENAI_API_KEY)  — when running in Colab
      2. Environment variable   OPENAI_API_KEY             — fallback / local runs

    To add the key in Colab:
      - Click the 🔑 key icon in the left sidebar
      - Add a secret named  OPENAI_API_KEY  with your key value
      - Toggle "Notebook access" ON
    """
    api_key = ""

    # 1️⃣  Try Colab Secrets first
    if _COLAB:
        try:
            api_key = colab_userdata.get("OPENAI_API_KEY").strip()
            logger.info("✅ API key loaded from Colab Secrets.")
        except Exception as e:
            logger.warning("Colab Secrets lookup failed (%s). Falling back to env var.", e)

    # 2️⃣  Fall back to environment variable
    if not api_key:
        api_key = os.environ.get("OPENAI_API_KEY", "").strip()
        if api_key:
            logger.info("✅ API key loaded from environment variable.")

    # 3️⃣  Neither source had a key — raise a clear error
    if not api_key:
        raise EnvironmentError(
            "OPENAI_API_KEY not found.\n"
            "• In Colab : open the 🔑 Secrets panel and add OPENAI_API_KEY.\n"
            "• Locally  : run  export OPENAI_API_KEY=sk-...  before launching."
        )

    if not api_key.startswith("sk-"):
        logger.warning("OPENAI_API_KEY does not start with 'sk-' — double-check the value.")

    return OpenAI(api_key=api_key)


# ── Prompt Builders ────────────────────────────────────────────────────────────
def build_prompts(source_language: str, target_language: str, code_snippet: str) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) with all placeholders resolved."""

    system_prompt = f"""
You are a compiler-level code translator and C++ expert.

Your output MUST compile successfully in a standard C++17 compiler.
If the code would not compile, you MUST fix it before output.

═══════════════════════════════
MANDATORY COMPILATION RULES
═══════════════════════════════

1. HEADER MANAGEMENT (STRICT)
You MUST include ALL required headers based on usage:

- std::vector        → #include <vector>
- std::transform     → #include <algorithm>
- std::accumulate    → #include <numeric>
- std::thread        → #include <thread>
- std::chrono        → #include <chrono>
- std::setprecision  → #include <iomanip>
- std::function      → #include <functional>
- std::optional      → #include <optional>

DO NOT omit ANY required header.

═══════════════════════════════
2. CRITICAL: VOID FUNCTION HANDLING
═══════════════════════════════

You MUST NEVER write:

    auto result = func(...);   ← FORBIDDEN if func returns void

Instead, you MUST use EXACTLY this pattern:

template <typename Func, typename... Args>
auto timer(Func&& func, Args&&... args) {{
    auto start = std::chrono::high_resolution_clock::now();

    if constexpr (std::is_void_v<std::invoke_result_t<Func, Args...>>) {{
        func(std::forward<Args>(args)...);

        auto end = std::chrono::high_resolution_clock::now();
        std::chrono::duration<double> elapsed = end - start;

        std::cout << "Execution time: " << elapsed.count() << " sec\\n";
    }} else {{
        auto result = func(std::forward<Args>(args)...);

        auto end = std::chrono::high_resolution_clock::now();
        std::chrono::duration<double> elapsed = end - start;

        std::cout << "Execution time: " << elapsed.count() << " sec\\n";
        return result;
    }}
}}

This rule is NON-NEGOTIABLE.

═══════════════════════════════
3. STANDARD LIBRARY USAGE
═══════════════════════════════

- Always prefix with std:: OR include using statements
- transform MUST be:
    std::transform(...)
- accumulate MUST be:
    std::accumulate(...)

═══════════════════════════════
4. PYTHON → C++ CONVERSION RULES
═══════════════════════════════

- list → std::vector
- sum() → std::accumulate
- map() → std::transform
- decorators → inline wrapper or function call (NOT templates unless safe)
- dynamic typing → explicit types
- None → nullptr or std::optional

═══════════════════════════════
5. PRE-OUTPUT VALIDATION (MANDATORY)
═══════════════════════════════

Before output, verify:

✔ All functions compile
✔ No missing headers
✔ No use of undefined symbols
✔ No "auto result" for void
✔ All std functions properly namespaced
✔ No template deduction errors

If ANY issue exists → FIX before output.

═══════════════════════════════
OUTPUT FORMAT (STRICT)
═══════════════════════════════

<converted_code language="{target_language}">
ONLY code
NO explanation
NO markdown
</converted_code>
"""

    user_prompt = f"""
Task:
Convert the following code from {source_language} to {target_language}.

STRICT REQUIREMENTS:
- Output must compile without errors.
- Handle all edge cases explicitly.
- Ensure correctness over stylistic preference.
- Do NOT generate pseudo-code.

Input Code ({source_language}):
{code_snippet}
"""

    return system_prompt, user_prompt


# ── Model Call ─────────────────────────────────────────────────────────────────
def call_model(system_prompt: str, user_prompt: str) -> str:
    """
    Call the OpenAI API and return the assistant's text response.
    Raises descriptive RuntimeError on failure.
    """
    client = get_client()

    try:
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
        )
    except AuthenticationError:
        raise RuntimeError(
            "❌ Invalid OpenAI API key.\n"
            "Please check your key at https://platform.openai.com/api-keys"
        )
    except RateLimitError:
        raise RuntimeError(
            "⏳ Rate limit or quota reached.\n"
            "Please check your usage at https://platform.openai.com/usage"
        )
    except APIConnectionError:
        raise RuntimeError("Could not connect to the OpenAI API. Check your network.")
    except APIError as e:
        if "insufficient_quota" in str(e) or "billing" in str(e).lower():
            raise RuntimeError(
                "💳 Your OpenAI credit balance is too low.\n"
                "Please visit https://platform.openai.com/settings/billing to add credits."
            )
        raise RuntimeError(f"OpenAI API error: {e}")

    # Extract text from the response
    return response.choices[0].message.content or ""


# ── Response Parser ────────────────────────────────────────────────────────────
def extract_converted_code(raw_response: str) -> str:
    """
    Pull the code out of the <converted_code ...>...</converted_code> tag.
    Falls back to returning the full response if the tag is absent.
    """
    match = re.search(
        r"<converted_code[^>]*>(.*?)</converted_code>",
        raw_response,
        re.DOTALL,
    )
    if match:
        return match.group(1).strip()
    logger.warning("Could not find <converted_code> tag in model response; returning raw output.")
    return raw_response.strip()


# ── Dynamic Textbox Helper ─────────────────────────────────────────────────────
def dynamic_update(text: str, min_lines: int = 5, max_lines: int = 60) -> gr.update:
    """Return a gr.update that sets value and auto-sizes the Textbox."""
    lines = text.splitlines()
    display_lines = sum(max(1, (len(l) // 100) + 1) for l in lines)
    clamped = max(min_lines, min(display_lines, max_lines))
    return gr.update(value=text, lines=clamped)


# ── Main Processing Function ───────────────────────────────────────────────────
def process_code(
    code_input: str,
    convert_from: str,
    convert_to: str,
) -> gr.update:
    """
    Gradio handler. Accepts raw code from the input Textbox,
    converts it using the GPT model, and returns a dynamic Textbox update.
    """

    # ── Input validation ───────────────────────────────────────────────────────
    if not code_input or not code_input.strip():
        return dynamic_update("⚠️  Input is empty. Please paste your source code and try again.")
    if not convert_from:
        return dynamic_update("⚠️  Please select a 'Convert From' language.")
    if not convert_to:
        return dynamic_update("⚠️  Please select a 'Convert To' language.")
    if convert_from == convert_to:
        return dynamic_update(
            "⚠️  'Convert From' and 'Convert To' are the same language. "
            "Please select different languages."
        )
    if len(code_input) > MAX_INPUT_CHARS:
        return dynamic_update(
            f"⚠️  Input is too large ({len(code_input):,} chars). "
            f"Maximum allowed is {MAX_INPUT_CHARS:,} chars."
        )

    # ── Build prompts & call model ─────────────────────────────────────────────
    try:
        system_prompt, user_prompt = build_prompts(convert_from, convert_to, code_input.strip())
        logger.info("Calling model: %s → %s (%d chars)", convert_from, convert_to, len(code_input))
        raw_response   = call_model(system_prompt, user_prompt)
        converted_code = extract_converted_code(raw_response)
        logger.info("Conversion complete (%d chars).", len(converted_code))
    except RuntimeError as e:
        return dynamic_update(f"❌ {e}")
    except Exception as e:
        logger.exception("Unexpected error during conversion.")
        return dynamic_update(f"❌ Unexpected error:\n{e}")

    return dynamic_update(converted_code)


# ── UI Layout ──────────────────────────────────────────────────────────────────
with gr.Blocks(title="Code Converter", theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # 🔄 Code Converter
        Paste your source code, choose the languages, and click **Convert**.
        The output area resizes automatically to fit the result.
        """
    )

    with gr.Row():
        # ── Left panel: dropdowns + input code ────────────────────────────────
        with gr.Column(scale=1):
            convert_from = gr.Dropdown(
                label="Convert From",
                choices=LANGUAGES,
                value=None,
                interactive=True,
            )
            convert_to = gr.Dropdown(
                label="Convert To",
                choices=LANGUAGES,
                value=None,
                interactive=True,
            )
            code_input = gr.Textbox(
                label="Source Code",
                lines=20,
                max_lines=60,
                interactive=True,
                placeholder="Paste your source code here…",
                show_copy_button=True,
            )
            submit_btn = gr.Button("Convert", variant="primary", size="lg")

        # ── Right panel: converted output ──────────────────────────────────────
        with gr.Column(scale=1):
            output_box = gr.Textbox(
                label="Converted Code",
                lines=20,
                max_lines=60,
                interactive=False,
                placeholder="Converted code will appear here…",
                show_copy_button=True,
            )

    # ── Event wiring ───────────────────────────────────────────────────────────
    submit_btn.click(
        fn=process_code,
        inputs=[code_input, convert_from, convert_to],
        outputs=output_box,
    )

    gr.Markdown(
        """<sub>
        🔑 <b>Colab users</b>: add <code>OPENAI_API_KEY</code> in the Secrets panel (left sidebar) and enable Notebook access.<br>
        💻 <b>Local users</b>: run <code>export OPENAI_API_KEY=sk-...</code> before launching.<br>
        🤖 Model in use: <code>gpt-4o</code> — change the <code>MODEL</code> constant at the top of the file to switch models.
        </sub>"""
    )


# In Colab, share=True creates a public tunnel URL.
# server_port is omitted so Gradio auto-selects a free port.
demo.launch(
    share=True,   # set False if running locally; share=True gives a public tunnel URL in Colab
)

In [ ]:
"""
Code Converter — Production-Grade Gradio App
Converts source code between languages using a HuggingFace open-source model.
Runs in Google Colab (GPU recommended) or locally.
"""

#--------------------------------------------------------------------
# This program uses the Hugging face Open source model by caling API
# ___________________________________________________________________


import re
import logging
import gradio as gr
import torch
from transformers import pipeline, Pipeline

# ── Colab detection ────────────────────────────────────────────────────────────
try:
    from google.colab import userdata as colab_userdata
    _COLAB = True
except ImportError:
    _COLAB = False

# ── Logging ────────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)
logger.info("Running in %s", "Google Colab" if _COLAB else "local environment")

# ── Constants ──────────────────────────────────────────────────────────────────
LANGUAGES       = ["Java", "Python", "Javascript", "C++", "COBOL", "VC++", ".NET"]
MODEL           = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_NEW_TOKENS  = 512       # raised — 1024 was too low for full code conversions
MAX_INPUT_CHARS = 100_000    # ~100 KB safety cap on pasted input


# ── Model Loading (once at startup) ───────────────────────────────────────────
def load_pipeline() -> Pipeline:
    """
    Load the text-generation pipeline onto the best available device.
    - CUDA GPU  → fast inference (recommended)
    - CPU       → slow but functional for the 1.5B model
    Raises RuntimeError with a clear message if loading fails.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype  = torch.float16 if device == "cuda" else torch.float32

    logger.info("Loading model '%s' on %s (dtype=%s)…", MODEL, device.upper(), dtype)

    try:
        pipe = pipeline(
            "text-generation",
            model=MODEL,
            torch_dtype=dtype,
            device_map="auto" if device == "cuda" else None,
        )
        logger.info("✅ Model loaded successfully on %s.", device.upper())
        return pipe
    except Exception as e:
        logger.exception("Failed to load model.")
        raise RuntimeError(
            f"Could not load model '{MODEL}'.\n"
            f"Reason: {e}\n\n"
            "Troubleshooting:\n"
            "• Run:  !pip install transformers torch accelerate\n"
            "• Enable GPU: Runtime → Change runtime type → T4 GPU\n"
            f"• Model page: https://huggingface.co/{MODEL}"
        )


# Load once at module level — shared across all Gradio sessions
try:
    PIPE = load_pipeline()
    MODEL_LOAD_ERROR = None
except RuntimeError as _load_err:
    PIPE = None
    MODEL_LOAD_ERROR = str(_load_err)
    logger.error("Model failed to load at startup: %s", MODEL_LOAD_ERROR)


# ── Prompt Builders ────────────────────────────────────────────────────────────
def build_messages(source_language: str, target_language: str, code_snippet: str) -> list[dict]:
    """
    Build a chat-format messages list for the pipeline.
    Using the chat format (list of dicts) ensures the model's chat template
    is applied correctly and the response is structured as assistant output.
    """
    system_prompt = f"""You are a compiler-level code translator and {target_language} expert.

Your output MUST compile and run correctly in a standard {target_language} environment.

MANDATORY RULES:
1. Include ALL required headers/imports for the target language.
2. Preserve 100% of the original logic and functionality.
3. Use idiomatic, production-quality {target_language} patterns.
4. Handle language-specific differences (typing, memory management, async behaviour).
5. Mark assumptions with: // ASSUMPTION: <text>
6. Mark limitations with: // LIMITATION: <text>
7. Do NOT omit any part of the code.
8. Do NOT hallucinate libraries or APIs.

OUTPUT FORMAT (STRICT):
<converted_code language="{target_language}">
ONLY raw code here — no markdown fences, no explanations
</converted_code>"""

    user_prompt = f"""Convert the following {source_language} code to {target_language}.

Requirements:
- Output must compile and run without errors.
- Preserve all logic exactly.
- Use idiomatic {target_language} style.
- Add inline comments where the translation is non-obvious.

Input Code ({source_language}):
{code_snippet}"""

    return [
        {"role": "system",  "content": system_prompt},
        {"role": "user",    "content": user_prompt},
    ]


# ── Model Call ─────────────────────────────────────────────────────────────────
def call_model(source_language: str, target_language: str, code_snippet: str) -> str:
    if PIPE is None:
        raise RuntimeError(f"Model is not loaded.\n{MODEL_LOAD_ERROR}")

    # ✅ Build a SINGLE STRING prompt (NOT chat format)
    system_prompt = f"""You are a compiler-level code translator and {target_language} expert.

Your output MUST compile and run correctly in a standard {target_language} environment.

Return ONLY code inside:
<converted_code></converted_code>
"""

    user_prompt = f"""Convert {source_language} to {target_language}:

{code_snippet}
"""

    full_prompt = system_prompt + "\n\n" + user_prompt

    logger.info("Running inference...")

    try:
        outputs = PIPE(
            full_prompt,                     # ✅ STRING input (FIXED)
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=PIPE.tokenizer.eos_token_id,
            return_full_text=False          # ✅ important
        )
    except Exception as e:
        raise RuntimeError(f"Inference failed: {e}")

    try:
        generated_text = outputs[0]["generated_text"]

        if not generated_text:
            raise RuntimeError("Empty model output")

        return generated_text.strip()

    except Exception as e:
        raise RuntimeError(f"Output parsing failed: {e} | Raw: {outputs}")


# ── Response Parser ────────────────────────────────────────────────────────────
def extract_converted_code(raw_response: str) -> str:
    """
    Extract code from inside <converted_code ...>...</converted_code>.
    Falls back to the full response if the tag is absent.
    """
    if not isinstance(raw_response, str):
        raw_response = str(raw_response)

    match = re.search(
        r"<converted_code[^>]*>(.*?)</converted_code>",
        raw_response,
        re.DOTALL,
    )
    if match:
        return match.group(1).strip()

    logger.warning("No <converted_code> tag found — returning raw model output.")
    return raw_response.strip()


# ── Dynamic Textbox Helper ─────────────────────────────────────────────────────
def dynamic_update(text: str, min_lines: int = 5, max_lines: int = 60) -> gr.update:
    """Return a gr.update that sets value and auto-sizes the Textbox height."""
    lines = text.splitlines()
    display_lines = sum(max(1, (len(l) // 100) + 1) for l in lines)
    clamped = max(min_lines, min(display_lines, max_lines))
    return gr.update(value=text, lines=clamped)


# ── Main Processing Function ───────────────────────────────────────────────────
def process_code(code_input: str, convert_from: str, convert_to: str) -> gr.update:
    """
    Gradio handler. Validates inputs, calls the model, parses output,
    and returns a dynamically sized Textbox update.
    """

    # ── Fail fast if model never loaded ───────────────────────────────────────
    if PIPE is None:
        return dynamic_update(f"❌ Model failed to load at startup.\n\n{MODEL_LOAD_ERROR}")

    # ── Input validation ───────────────────────────────────────────────────────
    if not code_input or not code_input.strip():
        return dynamic_update("⚠️  Input is empty. Please paste your source code and try again.")
    if not convert_from:
        return dynamic_update("⚠️  Please select a 'Convert From' language.")
    if not convert_to:
        return dynamic_update("⚠️  Please select a 'Convert To' language.")
    if convert_from == convert_to:
        return dynamic_update(
            "⚠️  'Convert From' and 'Convert To' are the same language. "
            "Please select different languages."
        )
    if len(code_input) > MAX_INPUT_CHARS:
        return dynamic_update(
            f"⚠️  Input is too large ({len(code_input):,} chars). "
            f"Maximum allowed is {MAX_INPUT_CHARS:,} chars."
        )

    # ── Run inference ──────────────────────────────────────────────────────────
    try:
        logger.info("Starting conversion: %s → %s (%d chars)", convert_from, convert_to, len(code_input))
        raw_response   = call_model(convert_from, convert_to, code_input.strip())
        converted_code = extract_converted_code(raw_response)
        logger.info("Conversion complete (%d chars output).", len(converted_code))
    except RuntimeError as e:
        return dynamic_update(f"❌ {e}")
    except Exception as e:
        logger.exception("Unexpected error during conversion.")
        return dynamic_update(f"❌ Unexpected error:\n{e}")

    return dynamic_update(converted_code)


# ── UI Layout ──────────────────────────────────────────────────────────────────
with gr.Blocks(title="Code Converter", theme=gr.themes.Soft()) as demo:

    if MODEL_LOAD_ERROR:
        gr.Markdown(
            f"> ⚠️ **Model failed to load.** Conversion will not work until resolved.\n"
            f"> ```\n> {MODEL_LOAD_ERROR[:400]}\n> ```"
        )

    gr.Markdown(
        f"""
        # 🔄 Code Converter
        Paste your source code, choose the languages, and click **Convert**.

        > 🤖 **Model:** `{MODEL}`
        > ⚡ **Device:** `{"GPU (CUDA)" if torch.cuda.is_available() else "CPU — inference will be slow"}`
        """
    )

    with gr.Row():
        # ── Left panel — inputs ───────────────────────────────────────────────
        with gr.Column(scale=1):
            convert_from = gr.Dropdown(
                label="Convert From",
                choices=LANGUAGES,
                value=None,
                interactive=True,
            )
            convert_to = gr.Dropdown(
                label="Convert To",
                choices=LANGUAGES,
                value=None,
                interactive=True,
            )
            code_input = gr.Textbox(
                label="Source Code",
                lines=20,
                max_lines=60,
                interactive=True,
                placeholder="Paste your source code here…",
                show_copy_button=True,
            )
            submit_btn = gr.Button(
                "Convert",
                variant="primary",
                size="lg",
                interactive=(PIPE is not None),
            )

        # ── Right panel — output ──────────────────────────────────────────────
        with gr.Column(scale=1):
            output_box = gr.Textbox(
                label="Converted Code",
                lines=20,
                max_lines=60,
                interactive=False,
                placeholder="Converted code will appear here…",
                show_copy_button=True,
            )

    submit_btn.click(
        fn=process_code,
        inputs=[code_input, convert_from, convert_to],
        outputs=output_box,
        show_progress=True   # ✅ ADD THIS
    )

    gr.Markdown(
        """<sub>
        💡 <b>Colab tip</b>: Enable a <b>T4 GPU</b> runtime for best performance
        (Runtime → Change runtime type → T4 GPU).<br>
        📦 <b>Install</b>: <code>!pip install transformers torch accelerate gradio</code><br>
        🔄 To switch models, change the <code>MODEL</code> constant at the top of the file.
        </sub>"""
    )


# ── Launch ─────────────────────────────────────────────────────────────────────
demo.launch(
    share=True,    # public tunnel URL — required in Colab
    debug=True,    # shows Python exceptions in the Gradio UI
)